### Setup

In [6]:
import pandas as pd
import os
from typhoon_ocr import ocr_document
from bs4 import BeautifulSoup
import re
from rapidfuzz import fuzz, process
from tqdm import tqdm

In [7]:
SUBMISSION_PATH = "../data/submission_template.csv"

In [8]:
template_df = pd.read_csv(SUBMISSION_PATH)
template_df

,id,doc_id,row_num,party_name,votes
0,constituency_10_1_1,constituency_10_1,1,ประชาธิปัตย์,0
1,constituency_10_1_2,constituency_10_1,2,ภูมิใจไทย,0
2,constituency_10_1_3,constituency_10_1,3,เศรษฐกิจ,0
3,constituency_10_1_4,constituency_10_1,4,กล้าธรรม,0
4,constituency_10_1_5,constituency_10_1,5,พลวัต,0
...,...,...,...,...,...
10048,party_list_34_11_53,party_list_34_11,53,ไทยพิทักษ์ธรรม,0
10049,party_list_34_11_54,party_list_34_11,54,ความหวังใหม่,0
10050,party_list_34_11_55,party_list_34_11,55,ไทยรวมไทย,0
10051,party_list_34_11_56,party_list_34_11,56,เพื่อบ้านเมือง,0


### EDA

In [9]:
template_df['party_name'].unique()

array(['ประชาธิปัตย์', 'ภูมิใจไทย', 'เศรษฐกิจ', 'กล้าธรรม', 'พลวัต',
       'ประชาชน', 'เพื่อไทย', 'ไทยภักดี', 'รวมไทยสร้างชาติ', 'ปวงชนไทย',
       'ไทยสร้างไทย', 'โอกาสใหม่', 'วิชชั่นใหม่', 'ประชาธิปไตยใหม่',
       'รักชาติ', 'ไทยก้าวใหม่', 'ทางเลือกใหม่', 'พลังประชารัฐ',
       'ประชากรไทย', 'ความหวังใหม่', 'อนาคตไทย', 'บ้านเมือง',
       'เพื่อบ้านเมือง', 'ไทยก้าวหน้า', 'รวมไทยสร้าง', 'แรงงานสร้างชาติ',
       'ไทยชนะ', 'พร้อม', 'ไทยพิทักษ์ธรรม', 'ไทยก้าวไทย', 'ไทยก้าวใหม',
       'เสรีรวมไทย', 'เป็นธรรม', 'ไทยธรรม', 'ฟิวชน', 'ฟิวชัน',
       'รวมพลังประชาชน', 'แรงงานสร้างไทย', 'ไทยสร้างชาติ', nan,
       'ปวงชนชาวไทย', 'กลาธรรม', 'สังคมประชาธิปไตยไทย', 'คลองไทย',
       'รวมใจไทย', 'รวมไทยสร้างชา', 'รักษ์ธรรม', 'ชาติ', 'ไทยพร้อม',
       'สร้างชาติ', 'ใหม่', 'พลังสังคมใหม่', 'สร้างอนาคตไทย',
       'ไทยทรัพย์ทวี', 'รวมพลัง', 'ไทรวมพลัง', 'เพื่อชาติไทย', 'มิติใหม่',
       'ท้องที่ไทย', 'พลังเพื่อไทย', 'ก้าวอิสระ', 'เพื่อชีวิตใหม่',
       'ครูไทยเพื่อประชาชน', 'ประชาชาติ', 'พลังธ

In [10]:
template_df.head()

,id,doc_id,row_num,party_name,votes
0,constituency_10_1_1,constituency_10_1,1,ประชาธิปัตย์,0
1,constituency_10_1_2,constituency_10_1,2,ภูมิใจไทย,0
2,constituency_10_1_3,constituency_10_1,3,เศรษฐกิจ,0
3,constituency_10_1_4,constituency_10_1,4,กล้าธรรม,0
4,constituency_10_1_5,constituency_10_1,5,พลวัต,0


In [11]:
invalid_name = ["Unknown Party", "พรรคที่ 1 (ไม่ระบุชื่อ)", "ไม่ระบุ"]

NaN_df = template_df[
    template_df['party_name'].isna() | template_df['party_name'].isin(invalid_name)
]

NaN_df

,id,doc_id,row_num,party_name,votes
737,constituency_14_2_10,constituency_14_2,10,NaN,0
738,constituency_14_2_11,constituency_14_2,11,NaN,0
739,constituency_14_2_12,constituency_14_2,12,NaN,0
740,constituency_14_2_13,constituency_14_2,13,NaN,0
741,constituency_14_2_14,constituency_14_2,14,NaN,0
742,constituency_14_2_15,constituency_14_2,15,NaN,0
743,constituency_14_2_16,constituency_14_2,16,NaN,0
744,constituency_14_2_17,constituency_14_2,17,NaN,0
2698,party_list_10_3_1,party_list_10_3,1,NaN,0
2700,party_list_10_3_3,party_list_10_3,3,NaN,0


In [12]:
# 737 - 744 Correctly Null
template_df.iloc[737]

id            constituency_14_2_10
doc_id           constituency_14_2
row_num                         10
party_name                     NaN
votes                            0
Name: 737, dtype: object

In [13]:
# 2698
template_df.iloc[2698]

id            party_list_10_3_1
doc_id          party_list_10_3
row_num                       1
party_name                  NaN
votes                         0
Name: 2698, dtype: object

In [14]:
# 2700
template_df.iloc[2700]

id            party_list_10_3_3
doc_id          party_list_10_3
row_num                       3
party_name                  NaN
votes                         0
Name: 2700, dtype: object

In [15]:
# 6461
template_df.iloc[6461]

id                  party_list_21_4_1
doc_id                party_list_21_4
row_num                             1
party_name    พรรคที่ 1 (ไม่ระบุชื่อ)
votes                               0
Name: 6461, dtype: object

In [16]:
# 6892
template_df.iloc[6892]

id            party_list_24_2_33
doc_id           party_list_24_2
row_num                       33
party_name                   NaN
votes                          0
Name: 6892, dtype: object

In [17]:
# 7031
template_df.iloc[7031]

id            party_list_25_1_1
doc_id          party_list_25_1
row_num                       1
party_name                  NaN
votes                         0
Name: 7031, dtype: object

In [18]:
# 7033
template_df.iloc[7033]

id            party_list_25_1_3
doc_id          party_list_25_1
row_num                       3
party_name                  NaN
votes                         0
Name: 7033, dtype: object

In [19]:
# 8000
template_df.iloc[8000]

id            party_list_30_4_1
doc_id          party_list_30_4
row_num                       1
party_name        Unknown Party
votes                         0
Name: 8000, dtype: object

In [20]:
# 9426
template_df.iloc[9426]

id            party_list_33_2_1
doc_id          party_list_33_2
row_num                       1
party_name                  NaN
votes                         0
Name: 9426, dtype: object

In [21]:
fix_invalid = {
    2698: ('ไทยทรัพย์ทวี', 31),
    2700: ('ใหม่', 149),
    6461: ('ไทยทรัพย์ทวี', 473),
    6892: ('ประชาชาติ', 632),
    7031: ('ไทยทรัพย์ทวี', 2250),
    7033: ('ใหม่', 481),
    8000: ('ไทยทรัพย์ทวี', 420),
    9426: ('ไทยทรัพย์ทวี', 347)
}

In [22]:
# Fix invalid
for idx, (party, vote) in fix_invalid.items():
    template_df.loc[idx, 'party_name'] = party
    template_df.loc[idx, 'votes'] = vote

for idx, (party, vote) in fix_invalid.items():
    print(template_df.iloc[idx])
    print()

id            party_list_10_3_1
doc_id          party_list_10_3
row_num                       1
party_name         ไทยทรัพย์ทวี
votes                        31
Name: 2698, dtype: object

id            party_list_10_3_3
doc_id          party_list_10_3
row_num                       3
party_name                 ใหม่
votes                       149
Name: 2700, dtype: object

id            party_list_21_4_1
doc_id          party_list_21_4
row_num                       1
party_name         ไทยทรัพย์ทวี
votes                       473
Name: 6461, dtype: object

id            party_list_24_2_33
doc_id           party_list_24_2
row_num                       33
party_name             ประชาชาติ
votes                        632
Name: 6892, dtype: object

id            party_list_25_1_1
doc_id          party_list_25_1
row_num                       1
party_name         ไทยทรัพย์ทวี
votes                      2250
Name: 7031, dtype: object

id            party_list_25_1_3
doc_id          party_list_2

### Extraction

In [23]:
def thai_num_to_int(text: str) -> int:
    # Step 1: convert Thai digits to Arabic digits
    thai_to_arabic = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    text = text.translate(thai_to_arabic)

    # Step 2: extract leading number (digits + optional commas)
    match = re.match(r"[\d,]+", text)
    if not match:
        raise ValueError("No numeric value found")

    numeric_str = match.group().replace(",", "")

    # Step 3: convert to int
    return int(numeric_str)

In [ ]:
def extract_party_score_dict(html: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")

    table = soup.find("table")
    if table is None:
        raise ValueError("No <table> found")

    rows = table.find_all("tr")
    if not rows:
        return {}

    # Extract header row
    headers = [td.get_text(strip=True) for td in rows[0].find_all(["td", "th"])]

    # Find best matching columns using Levenshtein-based similarity
    party_idx = None
    score_idx = None
    best_party_score = 0
    best_score_score = 0

    for i, h in enumerate(headers):
        party_sim = fuzz.partial_ratio(h, "พรรคการเมือง")
        score_sim = fuzz.partial_ratio(h, "ได้คะแนน")

        if party_sim > best_party_score:
            best_party_score = party_sim
            party_idx = i

        if score_sim > best_score_score:
            best_score_score = score_sim
            score_idx = i

    # Optional: enforce threshold to avoid wrong matches
    if best_party_score < 60 or best_score_score < 60:
        raise ValueError("Required columns not confidently found")

    result = {}

    for row in rows[1:]:
        cols = [td.get_text(strip=True) for td in row.find_all("td")]

        if len(cols) <= max(party_idx, score_idx):
            continue

        key = cols[party_idx]
        value = cols[score_idx]

        try:
            value = thai_num_to_int(value)
        except Exception as e:
            print(e.__str__())
            pass

        result[key] = value

    return result

In [ ]:
def extraction(path):
    try:
        markdown = ocr_document(
            pdf_or_image_path=path
        )
        party_dict = extract_party_score_dict(markdown)
        return party_dict
    except Exception as e:
        print(e.__str__())
        return {}

### Inference

In [26]:
def merge_pages(*pages):
    merged = {}
    for page in pages:
        if page is None:
            continue
        for k, v in page.items():
            merged[k] = merged.get(k, 0) + v
    return merged

In [27]:
def assign_votes(df, vote_dict, threshold=80):
    # Remove non-party keys
    vote_dict = {
        k: v for k, v in vote_dict.items()
        if k != 'รวมคะแนนทั้งสิ้น'
    }

    keys = list(vote_dict.keys())

    def get_vote(party_name):
        match = process.extractOne(
            party_name,
            keys,
            scorer=fuzz.ratio
        )
        
        if match is None:
            return 0
        
        best_key, score, _ = match
        
        if score >= threshold:
            return vote_dict[best_key]
        return 0

    df['votes'] = df['party_name'].apply(get_vote)
    return df

In [28]:
submission_df = template_df.copy()

In [ ]:
PREFIX = "../data/images/"

doc_ids = submission_df['doc_id'].unique()

for doc_id in tqdm(doc_ids, desc="Processing", unit="doc"):
    page1 = extraction(PREFIX + doc_id + '.png')
    page2 = extraction(PREFIX + doc_id + '_page2.png')
    page3 = extraction(PREFIX + doc_id + '_page3.png')
    page4 = extraction(PREFIX + doc_id + '_page4.png')

    vote_dict = merge_pages(page1, page2, page3, page4)

    mask = submission_df['doc_id'] == doc_id

    submission_df.loc[mask] = assign_votes(
        submission_df.loc[mask].copy(),
        vote_dict,
        threshold=80
    )

Processing:   0%|          | 0/300 [00:28<?, ?doc/s]


In [31]:
submission_df

,id,doc_id,row_num,party_name,votes
0,constituency_10_1_1,constituency_10_1,1,ประชาธิปัตย์,14813
1,constituency_10_1_2,constituency_10_1,2,ภูมิใจไทย,14368
2,constituency_10_1_3,constituency_10_1,3,เศรษฐกิจ,979
3,constituency_10_1_4,constituency_10_1,4,กล้าธรรม,244
4,constituency_10_1_5,constituency_10_1,5,พลวัต,351
...,...,...,...,...,...
10048,party_list_34_11_53,party_list_34_11,53,ไทยพิทักษ์ธรรม,0
10049,party_list_34_11_54,party_list_34_11,54,ความหวังใหม่,0
10050,party_list_34_11_55,party_list_34_11,55,ไทยรวมไทย,0
10051,party_list_34_11_56,party_list_34_11,56,เพื่อบ้านเมือง,0
